# Eksperimen 3 (revisi CMC, R1-5): rasio tambahan r = 0.08 dan r = 0.12

**Notebook ini sudah disetel `EXPERIMENT = "rasio"`, tidak perlu diubah.** Estimasi ~8,6 jam.

Satu notebook untuk tiga eksperimen. **Pilih satu dengan mengubah `EXPERIMENT` di bagian 3**, lalu Save & Run All. Jalankan satu eksperimen per sesi agar tidak melewati batas 12 jam.

| EXPERIMENT | Butir reviewer | Varian | Estimasi GPU |
|---|---|---|---|
| `"ablasi_r016"` | R1-6 | r = 0.16 tanpa ESE; r = 0.16 dengan blok LN asli (tanpa BN dan ESE) | ~8,6 jam |
| `"ablasi_r016_warmup"` | R1-6 (opsional) | r = 0.16 tanpa warmup | ~4,3 jam |
| `"repeat_cv"` | R1-7, R3-Q4 | kontrol dan r = 0.16 dengan **fold seed 2026** (pengulangan CV kedua) | ~8,6 jam |
| `"rasio"` | R1-5 | r = 0.08 dan r = 0.12 | ~8,6 jam |

Semua varian memakai protokol Tabel 3: 300 epoch, AdamW 3e-4, EMA 0.999, augmentasi geometric, stepped stem + BN + ESE + warmup 20, kecuali komponen yang sengaja dihapus. Pembanding (r = 0.16 lengkap dan kontrol, fold seed 42) dibaca dari `results_cv.csv` lama.

Catatan ablasi: di kode, ESE hanya tersedia pada blok BN, jadi varian "LN" otomatis tanpa ESE dan memakai LayerScale, sama seperti konfigurasi 1 di Tabel 5.

### Sebelum menjalankan
1. **Add Input → Datasets →** `batik-clean-v1`
2. **Add Input → Notebook Output →** notebook lama yang berisi `results_cv.csv` (hasil Tabel 7 dan 9)
3. Bila melanjutkan sesi yang terputus: **Add Input → Notebook Output →** versi sebelumnya dari notebook ini (fold yang selesai dilewati)
4. **Settings → Accelerator → GPU T4**, lalu **Save Version → Save & Run All**
5. Unduh dari tab Output: `results_revisi_cv.csv`, `ringkasan_<EXPERIMENT>.csv`, folder `preds/`

## 1. Tulis file model & scaffold pelatihan (tidak diubah dari notebook asli)

In [ ]:
%%writefile cspconvnext.py
"""
CSPConvNeXt / E-ConvNeXt untuk klasifikasi motif batik.

Mengadaptasi perbaikan dari:
  F. Wang et al., "E-ConvNeXt: combining CSPNet with ConvNeXt" (preprint, Elsevier)

Setiap perbaikan dibuat TOGGLEABLE agar bisa dilakukan studi ablasi per komponen:
  1. stem_type="stepped"  -> Stepped Stem (ganti patchify 4x4 stride-4)
  2. block_norm="bn"      -> BatchNorm2d + Conv1x1 (ganti LayerNorm + transpose + FC)
  3. use_ese=True         -> ESE channel attention (ganti LayerScale)
  4. skip_ratio           -> rasio split CSP (0.5 = split naif 50/50, versi "original")

Kombinasi "original CSPConvNeXt" (versi terlemah di ablasi paper):
    stem_type="patchify", block_norm="ln", use_ese=False, skip_ratio=0.5
Kombinasi "improved" (adaptasi E-ConvNeXt):
    stem_type="stepped", block_norm="bn", use_ese=True

Verifikasi lokal:
    python cspconvnext.py
"""

import torch
import torch.nn as nn


# ----------------------------------------------------------------------------
# Utilitas
# ----------------------------------------------------------------------------
class LayerNorm2d(nn.Module):
    """LayerNorm pada dimensi channel untuk tensor (B, C, H, W) (channels-first)."""

    def __init__(self, num_channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))
        self.eps = eps

    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return self.weight[None, :, None, None] * x + self.bias[None, :, None, None]


class DropPath(nn.Module):
    """Stochastic depth (drop path) per-sampel."""

    def __init__(self, p=0.0):
        super().__init__()
        self.p = p

    def forward(self, x):
        if self.p == 0.0 or not self.training:
            return x
        keep = 1 - self.p
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = keep + torch.rand(shape, dtype=x.dtype, device=x.device)
        mask.floor_()
        return x / keep * mask


class ESEBlock(nn.Module):
    """Effective Squeeze-and-Excitation: GAP -> satu Conv1x1 (tanpa reduksi) -> HardSigmoid.
    Dipilih paper karena akurasi tertinggi & tercepat dibanding SE/ECA/CBAM."""

    def __init__(self, channels):
        super().__init__()
        self.fc = nn.Conv2d(channels, channels, kernel_size=1)
        self.act = nn.Hardsigmoid()

    def forward(self, x):
        w = x.mean((2, 3), keepdim=True)
        return x * self.act(self.fc(w))


# ----------------------------------------------------------------------------
# Blok
# ----------------------------------------------------------------------------
class ConvNeXtBlockLN(nn.Module):
    """Blok ConvNeXt ASLI: dwconv7x7 -> LN(channels-last) -> FC -> GELU -> FC -> LayerScale."""

    def __init__(self, dim, drop_path=0.0, layer_scale_init=1e-6):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = (
            nn.Parameter(layer_scale_init * torch.ones(dim))
            if layer_scale_init > 0 else None
        )
        self.drop_path = DropPath(drop_path)

    def forward(self, x):
        inp = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)          # NCHW -> NHWC
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2)          # NHWC -> NCHW
        return inp + self.drop_path(x)


class ConvNeXtBlockBN(nn.Module):
    """Blok versi E-ConvNeXt: dwconv7x7 -> BN -> Conv1x1 -> GELU -> Conv1x1 -> BN -> ESE.
    Tanpa transpose (lebih cepat menurut paper). BN sebelum ESE mencegah
    ketidakstabilan/collapse saat channel attention ditambahkan."""

    def __init__(self, dim, drop_path=0.0, use_ese=True):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm1 = nn.BatchNorm2d(dim)
        self.pwconv1 = nn.Conv2d(dim, 4 * dim, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(4 * dim, dim, kernel_size=1)
        self.norm2 = nn.BatchNorm2d(dim)   # normalisasi sebelum attention (perbaikan paper)
        self.ese = ESEBlock(dim) if use_ese else None
        self.drop_path = DropPath(drop_path)

    def forward(self, x):
        inp = x
        x = self.dwconv(x)
        x = self.norm1(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.norm2(x)
        if self.ese is not None:
            x = self.ese(x)
        return inp + self.drop_path(x)


def make_block(dim, drop_path, block_norm, use_ese):
    if block_norm == "bn":
        return ConvNeXtBlockBN(dim, drop_path, use_ese=use_ese)
    return ConvNeXtBlockLN(dim, drop_path)


# ----------------------------------------------------------------------------
# Stage
# ----------------------------------------------------------------------------
class ConvNeXtStage(nn.Module):
    """Stage STANDAR: semua channel diproses seluruh blok (baseline non-CSP)."""

    def __init__(self, dim, depth, drop_paths, block_norm="ln", use_ese=False):
        super().__init__()
        self.blocks = nn.Sequential(
            *[make_block(dim, drop_paths[i], block_norm, use_ese) for i in range(depth)]
        )

    def forward(self, x):
        return self.blocks(x)


class CSPConvNeXtStage(nn.Module):
    """Stage CSP: sebagian channel dilewatkan langsung, sisanya diproses blok, lalu
    concat + transisi 1x1.

    skip_ratio: fraksi channel yang DILEWATKAN LANGSUNG (0.5 = split naif 50/50)."""

    def __init__(self, dim, depth, drop_paths, use_transition=True, skip_ratio=0.5,
                 block_norm="ln", use_ese=False):
        super().__init__()
        assert 0.0 < skip_ratio < 1.0, "skip_ratio harus di antara 0 dan 1"
        self.skip_ch = max(1, round(dim * skip_ratio))
        self.proc_ch = dim - self.skip_ch
        self.blocks = nn.Sequential(
            *[make_block(self.proc_ch, drop_paths[i], block_norm, use_ese)
              for i in range(depth)]
        )
        self.use_transition = use_transition
        if use_transition:
            self.transition = nn.Conv2d(dim, dim, kernel_size=1)

    def forward(self, x):
        part1, part2 = x[:, : self.skip_ch], x[:, self.skip_ch:]
        part2 = self.blocks(part2)
        out = torch.cat([part1, part2], dim=1)
        if self.use_transition:
            out = self.transition(out)
        return out


# ----------------------------------------------------------------------------
# Stem & downsample
# ----------------------------------------------------------------------------
class PatchifyStem(nn.Module):
    """Stem ConvNeXt asli: Conv 4x4 stride 4 (224 -> 56)."""

    def __init__(self, in_ch=3, dim=96, norm="ln"):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=4, stride=4)
        self.norm = nn.BatchNorm2d(dim) if norm == "bn" else LayerNorm2d(dim)

    def forward(self, x):
        return self.norm(self.proj(x))


class SteppedStem(nn.Module):
    """Stepped Stem (E-ConvNeXt): satu Conv 2x2 s2 + dua Conv 3x3, channel bertahap
    3 -> 32 -> 64 -> dim. Mengurangi kehilangan informasi akibat downsampling mendadak.
    Output spasial sama dengan patchify (224 -> 56)."""

    def __init__(self, in_ch=3, dim=96, norm="bn"):
        super().__init__()

        def nl(c):
            return nn.BatchNorm2d(c) if norm == "bn" else LayerNorm2d(c)

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=2, stride=2, padding=0),  # 224 -> 112
            nl(32), nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),     # 112 -> 56
            nl(64), nn.GELU(),
            nn.Conv2d(64, dim, kernel_size=3, stride=1, padding=1),    # 56 -> 56
            nl(dim),
        )

    def forward(self, x):
        return self.stem(x)


class Downsample(nn.Module):
    """Norm + Conv 2x2 stride 2 (H,W -> H/2,W/2 ; channel in->out)."""

    def __init__(self, in_dim, out_dim, norm="ln"):
        super().__init__()
        self.norm = nn.BatchNorm2d(in_dim) if norm == "bn" else LayerNorm2d(in_dim)
        self.reduce = nn.Conv2d(in_dim, out_dim, kernel_size=2, stride=2)

    def forward(self, x):
        return self.reduce(self.norm(x))


# ----------------------------------------------------------------------------
# Model penuh
# ----------------------------------------------------------------------------
class CSPConvNeXt(nn.Module):
    def __init__(
        self,
        num_classes,
        dims=(96, 192, 384, 768),
        depths=(3, 3, 9, 3),
        drop_path_rate=0.1,
        csp=True,                 # True = CSP; False = ConvNeXt standar (baseline ablasi)
        skip_ratio=0.5,           # fraksi channel dilewatkan langsung (hanya bila csp=True)
        use_transition=True,
        stem_type="stepped",      # "stepped" (E-ConvNeXt) | "patchify" (ConvNeXt asli)
        block_norm="bn",          # "bn" (E-ConvNeXt) | "ln" (ConvNeXt asli)
        use_ese=True,             # ESE channel attention (berlaku bila block_norm="bn")
        head_hidden=512,
        head_dropout=0.2,
    ):
        super().__init__()
        assert len(dims) == 4 and len(depths) == 4
        assert stem_type in ("stepped", "patchify")
        assert block_norm in ("bn", "ln")
        self.cfg_summary = dict(csp=csp, skip_ratio=skip_ratio, stem_type=stem_type,
                                block_norm=block_norm, use_ese=use_ese)

        if stem_type == "stepped":
            self.stem = SteppedStem(3, dims[0], norm=block_norm)
        else:
            self.stem = PatchifyStem(3, dims[0], norm=block_norm)

        total = sum(depths)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, total)]

        self.stages = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        cur = 0
        for i in range(4):
            dp = dpr[cur:cur + depths[i]]
            if csp:
                stage = CSPConvNeXtStage(dims[i], depths[i], dp, use_transition,
                                         skip_ratio, block_norm, use_ese)
            else:
                stage = ConvNeXtStage(dims[i], depths[i], dp, block_norm, use_ese)
            self.stages.append(stage)
            cur += depths[i]
            if i < 3:
                self.downsamples.append(Downsample(dims[i], dims[i + 1], norm=block_norm))

        self.norm = nn.LayerNorm(dims[-1], eps=1e-6)
        if head_hidden:
            self.head = nn.Sequential(
                nn.Linear(dims[-1], head_hidden),
                nn.GELU(),
                nn.Dropout(head_dropout),
                nn.Linear(head_hidden, num_classes),
            )
        else:
            self.head = nn.Linear(dims[-1], num_classes)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        for i in range(4):
            x = self.stages[i](x)
            if i < 3:
                x = self.downsamples[i](x)
        x = x.mean([-2, -1])   # Global Average Pooling
        x = self.norm(x)
        return self.head(x)


# ----------------------------------------------------------------------------
# Helper metrik
# ----------------------------------------------------------------------------
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_flops(model, input_size=(1, 3, 224, 224)):
    try:
        from thop import profile
        model.eval()
        inp = torch.randn(*input_size)
        flops, _ = profile(model, inputs=(inp,), verbose=False)
        return flops
    except ImportError:
        return None


# ----------------------------------------------------------------------------
# Self-test: ablasi tiap perbaikan
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    N_CLASSES = 20
    x = torch.randn(2, 3, 224, 224)

    configs = [
        ("ConvNeXt baseline (LN, patchify)",
         dict(csp=False, stem_type="patchify", block_norm="ln", use_ese=False)),
        ("CSP original (LN, patchify, 50/50)",
         dict(csp=True, skip_ratio=0.5, stem_type="patchify", block_norm="ln", use_ese=False)),
        ("CSP + BN block",
         dict(csp=True, skip_ratio=0.5, stem_type="patchify", block_norm="bn", use_ese=False)),
        ("CSP + BN + ESE",
         dict(csp=True, skip_ratio=0.5, stem_type="patchify", block_norm="bn", use_ese=True)),
        ("CSP + BN + ESE + Stepped (full)",
         dict(csp=True, skip_ratio=0.5, stem_type="stepped", block_norm="bn", use_ese=True)),
        ("Full, skip_ratio=0.25",
         dict(csp=True, skip_ratio=0.25, stem_type="stepped", block_norm="bn", use_ese=True)),
    ]
    for name, kwargs in configs:
        model = CSPConvNeXt(num_classes=N_CLASSES, **kwargs)
        model.eval()
        with torch.no_grad():
            y = model(x)
        assert y.shape == (2, N_CLASSES), f"bentuk output salah: {y.shape}"
        params = count_params(model) / 1e6
        flops = count_flops(model)
        fs = f"{flops / 1e9:5.2f} GFLOPs" if flops else "n/a"
        print(f"{name:36s} | {params:6.2f} M params | {fs}")

    print("\nSanity check lolos: semua konfigurasi forward pass OK.")

In [ ]:
%%writefile batik_train.py
"""
batik_train.py — Scaffold pelatihan & evaluasi (PyTorch + timm).

Dua mode evaluasi:
  1. train_one(cfg)   -> split tetap train/val/test (cara lama)
  2. run_kfold(cfg)   -> stratified k-fold CV pada SELURUH gambar (disarankan)

Perbaikan stabilitas (mengatasi variansi antar-run):
  - ckpt_select="ema"  : bobot EMA (exponential moving average) dipakai utk evaluasi.
                         Menghilangkan kerapuhan "pilih checkpoint terbaik dari val"
                         yang sangat noisy saat val set kecil.
  - ckpt_select="last" : bobot epoch terakhir.
  - ckpt_select="best_val": cara lama (rapuh pada dataset kecil).

Pada mode k-fold, TIDAK ada val set: tiap fold melatih pada k-1 bagian dan menguji
pada 1 bagian, sehingga setiap gambar diuji tepat sekali (766 sampel uji efektif,
bukan 78). Estimasi jauh lebih presisi.

Butuh: pip install torch torchvision timm scikit-learn thop
Perlu cspconvnext.py di direktori yang sama.
"""

import os
import csv
import copy
import time
import random
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score)
from sklearn.model_selection import StratifiedKFold

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


# ============================ KONFIGURASI ============================
@dataclass
class Config:
    data_root: str = "/kaggle/input/batik-clean-v1/batik_clean"
    num_classes: int = 20
    img_size: int = 224
    batch_size: int = 32
    epochs: int = 300
    lr: float = 3e-4
    weight_decay: float = 0.05
    optimizer: str = "adamw"
    scheduler: str = "cosine"
    label_smoothing: float = 0.1
    drop_path: float = 0.1
    # --- augmentasi ---
    aug_geometric: bool = True
    aug_mixup_cutmix: bool = False
    # --- model ---
    model_name: str = "cspconvnext"
    csp_skip_ratio: float = 0.5
    stem_type: str = "stepped"
    block_norm: str = "bn"
    use_ese: bool = True
    pretrained: bool = False
    # --- resep pelatihan ---
    warmup_epochs: int = 20
    scale_lr_by_batch: bool = False
    # --- stabilisasi checkpoint (perbaikan variansi) ---
    ckpt_select: str = "ema"      # "ema" | "last" | "best_val"
    ema_decay: float = 0.999
    # --- k-fold ---
    n_folds: int = 5
    fold: int = -1                # -1 = semua fold (dipakai run_kfold)
    # --- lain-lain ---
    seed: int = 42
    num_workers: int = 2
    results_csv: str = "/kaggle/working/results.csv"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
# ====================================================================


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


# --------------------------- EMA ---------------------------
class ModelEMA:
    """Exponential moving average bobot model. Mengurangi variansi hasil secara drastis
    karena evaluasi tidak lagi bergantung pada satu checkpoint yang kebetulan bagus."""

    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        for e, m in zip(self.ema.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point:
                e.mul_(self.decay).add_(m.detach(), alpha=1.0 - self.decay)
            else:
                e.copy_(m)


# --------------------------- DATA ---------------------------
class ListImageDataset(Dataset):
    """Dataset dari daftar path + label (dipakai k-fold)."""

    def __init__(self, files, labels, transform):
        self.files, self.labels, self.transform = files, labels, transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")
        return self.transform(img), int(self.labels[i])


def build_transforms(cfg):
    if cfg.aug_geometric:
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(cfg.img_size, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomVerticalFlip(0.5),
            transforms.RandomRotation(20),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    else:
        train_tf = transforms.Compose([
            transforms.Resize(int(cfg.img_size * 1.14)),
            transforms.CenterCrop(cfg.img_size),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(cfg.img_size * 1.14)),
        transforms.CenterCrop(cfg.img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_tf, eval_tf


def pool_all_images(data_root):
    """Gabungkan SEMUA gambar dari train/val/test menjadi satu pool (untuk k-fold)."""
    splits = [s for s in ("train", "val", "test")
              if os.path.isdir(os.path.join(data_root, s))]
    classes = sorted({c for s in splits
                      for c in os.listdir(os.path.join(data_root, s))
                      if os.path.isdir(os.path.join(data_root, s, c))})
    cls2idx = {c: i for i, c in enumerate(classes)}
    files, labels = [], []
    for s in splits:
        for c in sorted(os.listdir(os.path.join(data_root, s))):
            d = os.path.join(data_root, s, c)
            if not os.path.isdir(d):
                continue
            for f in sorted(os.listdir(d)):
                if f.lower().endswith(IMG_EXTS):
                    files.append(os.path.join(d, f))
                    labels.append(cls2idx[c])
    return np.array(files), np.array(labels), classes


def get_loaders(cfg):
    """Loader untuk split tetap (mode lama)."""
    train_tf, eval_tf = build_transforms(cfg)
    train_ds = datasets.ImageFolder(os.path.join(cfg.data_root, "train"), train_tf)
    val_ds = datasets.ImageFolder(os.path.join(cfg.data_root, "val"), eval_tf)
    test_ds = datasets.ImageFolder(os.path.join(cfg.data_root, "test"), eval_tf)
    assert train_ds.class_to_idx == val_ds.class_to_idx == test_ds.class_to_idx
    g = torch.Generator(); g.manual_seed(cfg.seed)
    tl = DataLoader(train_ds, cfg.batch_size, shuffle=True, drop_last=True,
                    num_workers=cfg.num_workers, pin_memory=True, generator=g)
    vl = DataLoader(val_ds, cfg.batch_size, shuffle=False,
                    num_workers=cfg.num_workers, pin_memory=True)
    sl = DataLoader(test_ds, cfg.batch_size, shuffle=False,
                    num_workers=cfg.num_workers, pin_memory=True)
    return tl, vl, sl, train_ds.classes


# --------------------------- MODEL ---------------------------
def build_model(cfg):
    name = cfg.model_name.lower()
    if name in ("cspconvnext", "convnext_ours"):
        from cspconvnext import CSPConvNeXt
        return CSPConvNeXt(num_classes=cfg.num_classes,
                           csp=(name == "cspconvnext"),
                           skip_ratio=cfg.csp_skip_ratio,
                           stem_type=cfg.stem_type,
                           block_norm=cfg.block_norm,
                           use_ese=cfg.use_ese,
                           drop_path_rate=cfg.drop_path)
    import timm
    try:
        return timm.create_model(name, pretrained=cfg.pretrained,
                                 num_classes=cfg.num_classes, drop_path_rate=cfg.drop_path)
    except TypeError:
        return timm.create_model(name, pretrained=cfg.pretrained, num_classes=cfg.num_classes)


def measure_efficiency(model, img_size=224):
    params = sum(p.numel() for p in model.parameters()) / 1e6
    gflops = None
    try:
        from thop import profile
        macs, _ = profile(model.to("cpu").eval(),
                          inputs=(torch.randn(1, 3, img_size, img_size),), verbose=False)
        gflops = macs / 1e9
    except Exception as e:
        print(f"[WARN] gagal hitung FLOPs: {e}")
    return params, gflops


# --------------------------- EVALUASI ---------------------------
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    probs, trues = [], []
    for x, y in loader:
        p = torch.softmax(model(x.to(device)), dim=1).cpu().numpy()
        probs.append(p); trues.append(np.asarray(y))
    y_prob = np.concatenate(probs); y_true = np.concatenate(trues)
    y_pred = y_prob.argmax(1)
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    try:
        m["auc_macro"] = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        m["auc_macro"] = float("nan")
    return m, (y_true, y_pred, y_prob)


# --------------------------- OPTIMIZER / SCHEDULER ---------------------------
def _effective_lr(cfg):
    if cfg.scale_lr_by_batch:
        return 1.25e-4 * (cfg.batch_size / 128)
    return cfg.lr


def _make_optimizer(cfg, model):
    lr = _effective_lr(cfg)
    if cfg.optimizer == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg.weight_decay)
    return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                           weight_decay=cfg.weight_decay, nesterov=True)


def _make_scheduler(cfg, optimizer):
    from torch.optim.lr_scheduler import (CosineAnnealingLR, StepLR, LinearLR,
                                          SequentialLR, ConstantLR)
    warm = min(cfg.warmup_epochs, max(0, cfg.epochs - 1))
    if cfg.scheduler == "cosine":
        main = CosineAnnealingLR(optimizer, T_max=max(1, cfg.epochs - warm))
    elif cfg.scheduler == "step":
        main = StepLR(optimizer, step_size=max(1, cfg.epochs // 3), gamma=0.1)
    else:
        if warm <= 0:
            return None
        main = ConstantLR(optimizer, factor=1.0, total_iters=0)
    if warm > 0:
        warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warm)
        return SequentialLR(optimizer, [warmup, main], milestones=[warm])
    return main


# --------------------------- LOOP PELATIHAN INTI ---------------------------
def _train_loop(cfg, model, train_loader, val_loader=None, verbose=False):
    """Melatih model; mengembalikan model siap-evaluasi sesuai cfg.ckpt_select."""
    device = cfg.device
    mixup_fn, criterion = None, nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    if cfg.aug_mixup_cutmix:
        from timm.data import Mixup
        from timm.loss import SoftTargetCrossEntropy
        mixup_fn = Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=1.0, switch_prob=0.5,
                         mode="batch", label_smoothing=cfg.label_smoothing,
                         num_classes=cfg.num_classes)
        criterion = SoftTargetCrossEntropy()

    optimizer = _make_optimizer(cfg, model)
    scheduler = _make_scheduler(cfg, optimizer)
    ema = ModelEMA(model, cfg.ema_decay) if cfg.ckpt_select == "ema" else None

    best_f1, best_state = -1.0, None
    for epoch in range(cfg.epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            if mixup_fn is not None:
                x, y = mixup_fn(x, y)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()
            if ema is not None:
                ema.update(model)
        if scheduler:
            scheduler.step()
        if cfg.ckpt_select == "best_val" and val_loader is not None:
            vm, _ = evaluate(model, val_loader, device)
            if vm["f1_macro"] > best_f1:
                best_f1 = vm["f1_macro"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if verbose and (epoch + 1) % 25 == 0:
            print(f"  epoch {epoch+1}/{cfg.epochs}", flush=True)

    if cfg.ckpt_select == "ema":
        return ema.ema
    if cfg.ckpt_select == "best_val" and best_state is not None:
        model.load_state_dict(best_state)
    return model


def _append_row(path, row):
    exists = os.path.exists(path)
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            w.writeheader()
        w.writerow(row)


def _base_row(cfg):
    return {
        "model": cfg.model_name,
        "csp_skip_ratio": cfg.csp_skip_ratio if cfg.model_name == "cspconvnext" else None,
        "stem": cfg.stem_type, "block_norm": cfg.block_norm, "ese": cfg.use_ese,
        "warmup": cfg.warmup_epochs, "ckpt": cfg.ckpt_select,
        "epochs": cfg.epochs, "lr": round(_effective_lr(cfg), 8),
        "aug_geom": cfg.aug_geometric, "aug_mix": cfg.aug_mixup_cutmix,
        "seed": cfg.seed,
    }


# --------------------------- MODE 1: split tetap ---------------------------
def train_one(cfg, verbose=False):
    set_seed(cfg.seed)
    train_loader, val_loader, test_loader, classes = get_loaders(cfg)
    model = build_model(cfg).to(cfg.device)
    t0 = time.time()
    model = _train_loop(cfg, model, train_loader, val_loader, verbose)
    train_time = time.time() - t0

    test_metrics, arrays = evaluate(model, test_loader, cfg.device)
    _, eval_tf = build_transforms(cfg)
    tr_eval = DataLoader(datasets.ImageFolder(os.path.join(cfg.data_root, "train"), eval_tf),
                         cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    train_metrics, _ = evaluate(model, tr_eval, cfg.device)

    params, gflops = measure_efficiency(build_model(cfg), cfg.img_size)
    row = {**_base_row(cfg), "fold": None,
           "train_accuracy": round(train_metrics["accuracy"], 4),
           **{f"test_{k}": round(v, 4) for k, v in test_metrics.items()},
           "params_M": round(params, 3),
           "gflops": round(gflops, 3) if gflops else None,
           "train_time_s": round(train_time, 1)}
    _append_row(cfg.results_csv, row)
    if verbose:
        print(row)
    return row, arrays, classes


# --------------------------- MODE 2: k-fold CV ---------------------------
def run_kfold(cfg, verbose=True):
    """Stratified k-fold CV pada SELURUH gambar. Tiap gambar diuji tepat sekali."""
    files, labels, classes = pool_all_images(cfg.data_root)
    if verbose:
        print(f"Pool: {len(files)} gambar, {len(classes)} kelas, {cfg.n_folds}-fold")

    train_tf, eval_tf = build_transforms(cfg)
    skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
    params, gflops = measure_efficiency(build_model(cfg), cfg.img_size)

    all_true, all_pred, fold_rows = [], [], []
    for k, (tr_idx, te_idx) in enumerate(skf.split(files, labels)):
        if cfg.fold >= 0 and k != cfg.fold:
            continue
        set_seed(cfg.seed + k)
        tr_ds = ListImageDataset(files[tr_idx], labels[tr_idx], train_tf)
        te_ds = ListImageDataset(files[te_idx], labels[te_idx], eval_tf)
        g = torch.Generator(); g.manual_seed(cfg.seed + k)
        tl = DataLoader(tr_ds, cfg.batch_size, shuffle=True, drop_last=True,
                        num_workers=cfg.num_workers, pin_memory=True, generator=g)
        el = DataLoader(te_ds, cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=True)

        model = build_model(cfg).to(cfg.device)
        t0 = time.time()
        model = _train_loop(cfg, model, tl, None, verbose=False)
        dt = time.time() - t0

        m, (yt, yp, _) = evaluate(model, el, cfg.device)
        all_true.append(yt); all_pred.append(yp)
        row = {**_base_row(cfg), "fold": k,
               "train_accuracy": None,
               **{f"test_{kk}": round(v, 4) for kk, v in m.items()},
               "params_M": round(params, 3),
               "gflops": round(gflops, 3) if gflops else None,
               "train_time_s": round(dt, 1)}
        _append_row(cfg.results_csv, row)
        fold_rows.append(row)
        if verbose:
            print(f"  fold {k}: acc={m['accuracy']:.4f} f1={m['f1_macro']:.4f} "
                  f"({dt/60:.1f} menit)", flush=True)

    if not fold_rows:
        return None, None
    accs = [r["test_accuracy"] for r in fold_rows]
    f1s = [r["test_f1_macro"] for r in fold_rows]
    summary = {
        "model": cfg.model_name, "n_folds": len(fold_rows),
        "acc_mean": round(float(np.mean(accs)), 4), "acc_std": round(float(np.std(accs, ddof=1)) if len(accs) > 1 else 0.0, 4),
        "f1_mean": round(float(np.mean(f1s)), 4), "f1_std": round(float(np.std(f1s, ddof=1)) if len(f1s) > 1 else 0.0, 4),
        "params_M": round(params, 3), "gflops": round(gflops, 3) if gflops else None,
    }
    if verbose:
        print(f"  >>> {cfg.model_name}: acc {summary['acc_mean']:.4f}+/-{summary['acc_std']:.4f} | "
              f"f1 {summary['f1_mean']:.4f}+/-{summary['f1_std']:.4f}", flush=True)
    pooled = (np.concatenate(all_true), np.concatenate(all_pred)) if all_true else None
    return summary, pooled


def sanity_check(cfg):
    set_seed(cfg.seed)
    train_loader, *_ = get_loaders(cfg)
    model = build_model(cfg).to(cfg.device)
    x, y = next(iter(train_loader))
    x, y = x.to(cfg.device), y.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for i in range(60):
        opt.zero_grad()
        out = model(x)
        crit(out, y).backward(); opt.step()
        if (i + 1) % 15 == 0:
            print(f"step {i+1:3d} | batch_acc {(out.argmax(1) == y).float().mean():.3f}")


def plot_confusion(pooled, classes, out="/kaggle/working/confusion.png"):
    import matplotlib.pyplot as plt
    y_true, y_pred = pooled
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=90, fontsize=7)
    ax.set_yticklabels(classes, fontsize=7)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.colorbar(im); plt.tight_layout(); plt.savefig(out, dpi=150)
    print("disimpan:", out)


if __name__ == "__main__":
    print("Gunakan run_kfold(cfg) untuk evaluasi k-fold, atau train_one(cfg) untuk split tetap.")

## 2. Setup: install, pulihkan hasil, cari dataset, info CPU

In [ ]:
import os, glob, shutil, subprocess, sys, importlib, platform
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "thop"], check=False)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
import batik_train as bt
importlib.reload(bt)

RESULTS = "/kaggle/working/results_revisi_cv.csv"   # hasil baru (terpisah dari CSV lama)
OLD_RESULTS = "/kaggle/working/results_cv_lama.csv"
PRED_DIR = "/kaggle/working/preds"
os.makedirs(PRED_DIR, exist_ok=True)

# pulihkan CSV hasil baru (bila melanjutkan) dan CSV lama (pembanding regime geom)
for name, dst in [("results_revisi_cv.csv", RESULTS), ("results_cv.csv", OLD_RESULTS)]:
    if not os.path.exists(dst):
        found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if found:
            shutil.copy(found[0], dst); print(f"{name} dipulihkan dari: {found[0]}")
        else:
            print(f"[info] {name} tidak ditemukan di input.")
for p in glob.glob("/kaggle/input/**/preds/*.npz", recursive=True):
    dst = os.path.join(PRED_DIR, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst)

DATA_ROOT = None
for r, d, f in os.walk("/kaggle/input", followlinks=True):
    if {"train", "val", "test"}.issubset(set(d)):
        DATA_ROOT = r; break
if not DATA_ROOT:
    print("Isi /kaggle/input saat ini:")
    for r, d, f in os.walk("/kaggle/input"):
        if r.count(os.sep) <= 5: print("  ", r, d[:6])
assert DATA_ROOT, "Dataset batik-clean-v1 tidak ditemukan. Klik Add Input, cari batik-clean-v1, lalu jalankan ulang."
print("DATA_ROOT:", DATA_ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA - aktifkan GPU!")

# --- info perangkat keras (untuk placeholder Bagian 3.8) ---
cpu = subprocess.run("lscpu | grep -E 'Model name|^CPU\\(s\\)|Thread\\(s\\) per core'", shell=True,
                     capture_output=True, text=True).stdout
print(cpu, "torch threads:", torch.get_num_threads())

## 3. Pilih eksperimen

In [ ]:
EXPERIMENT = "rasio"   # sudah disetel untuk r0.08 dan r0.12; "ablasi_r016" | "ablasi_r016_warmup" | "repeat_cv" | "rasio"
N_FOLDS, EPOCHS = 5, 300

FULL = dict(model_name="cspconvnext", csp_skip_ratio=0.16, stem_type="stepped",
            block_norm="bn", use_ese=True, warmup_epochs=20, seed=42)
PRESETS = {
    "ablasi_r016": [
        ("r0.16_noESE", {**FULL, "use_ese": False}),
        ("r0.16_LN",    {**FULL, "block_norm": "ln", "use_ese": False}),
    ],
    "ablasi_r016_warmup": [
        ("r0.16_noWarmup", {**FULL, "warmup_epochs": 0}),
    ],
    "repeat_cv": [
        ("control_seed2026", {**FULL, "model_name": "convnext_ours", "seed": 2026}),
        ("r0.16_seed2026",   {**FULL, "seed": 2026}),
    ],
    "rasio": [
        ("r0.08", {**FULL, "csp_skip_ratio": 0.08}),
        ("r0.12", {**FULL, "csp_skip_ratio": 0.12}),
    ],
}
# pembanding dari CSV lama (fold seed 42); untuk repeat_cv pembanding adalah pasangan di dalam preset
REFERENCE = {"ablasi_r016": "r0.16_full", "ablasi_r016_warmup": "r0.16_full",
             "repeat_cv": "control_seed2026", "rasio": "control"}[EXPERIMENT]
VARIANTS = PRESETS[EXPERIMENT]

base = bt.Config(data_root=DATA_ROOT, num_classes=20, epochs=EPOCHS, n_folds=N_FOLDS,
                 ckpt_select="ema", ema_decay=0.999, aug_geometric=True, aug_mixup_cutmix=False,
                 results_csv=RESULTS)
files, labels, classes = bt.pool_all_images(DATA_ROOT)
print(f"Pool: {len(files)} gambar, {len(classes)} kelas")
assert len(files) == 766, "Jumlah gambar bukan 766: pastikan dataset batik-clean-v1 yang sama"
for label, ov in VARIANTS:
    p, g = bt.measure_efficiency(bt.build_model(bt.Config(**{**bt.asdict(base), **ov})))
    print(f"{label:18s} {p:6.2f} M params | {g:.3f} GFLOPs")

## 4. Jalankan (bagian paling lama)

In [ ]:
def done_folds(label):
    if not os.path.exists(RESULTS):
        return set()
    df = pd.read_csv(RESULTS)
    return set(df[df["variant"] == label]["fold"].astype(int))

for label, ov in VARIANTS:
    cfg = bt.Config(**{**bt.asdict(base), **ov})
    train_tf, eval_tf = bt.build_transforms(cfg)
    splits = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                  random_state=cfg.seed).split(files, labels))
    params, gflops = bt.measure_efficiency(bt.build_model(cfg))
    todo = [k for k in range(N_FOLDS) if k not in done_folds(label)]
    print(f"\n### {label}: dikerjakan fold {todo or '-'}", flush=True)
    for k in todo:
        tr_idx, te_idx = splits[k]
        bt.set_seed(cfg.seed + k)                 # identik dengan bt.run_kfold
        g = torch.Generator(); g.manual_seed(cfg.seed + k)
        tl = DataLoader(bt.ListImageDataset(files[tr_idx], labels[tr_idx], train_tf), cfg.batch_size,
                        shuffle=True, drop_last=True, num_workers=cfg.num_workers, pin_memory=True, generator=g)
        el = DataLoader(bt.ListImageDataset(files[te_idx], labels[te_idx], eval_tf), cfg.batch_size,
                        shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
        model = bt.build_model(cfg).to(cfg.device)
        t0 = bt.time.time()
        model = bt._train_loop(cfg, model, tl, None, verbose=False)
        dt = bt.time.time() - t0
        m, (yt, yp, prob) = bt.evaluate(model, el, cfg.device)
        np.savez_compressed(f"{PRED_DIR}/{label}_fold{k}.npz",
                            test_idx=te_idx, files=files[te_idx], y_true=yt, y_prob=prob)
        row = {**bt._base_row(cfg), "experiment": EXPERIMENT, "variant": label, "fold": k,
               **{f"test_{kk}": round(v, 4) for kk, v in m.items()},
               "params_M": round(params, 3), "gflops": round(gflops, 3), "train_time_s": round(dt, 1)}
        bt._append_row(RESULTS, row)
        print(f"  fold {k}: acc={m['accuracy']:.4f} f1={m['f1_macro']:.4f} ({dt/60:.1f} menit)", flush=True)
print("\nEksperimen selesai.")

## 5. Ringkasan dan uji berpasangan

In [ ]:
from scipy import stats
new = pd.read_csv(RESULTS)
new = new[new["variant"].isin([v for v, _ in VARIANTS])][["variant", "fold", "test_accuracy", "test_f1_macro", "params_M", "gflops"]]
frames = [new]

if REFERENCE in ("r0.16_full", "control") and os.path.exists(OLD_RESULTS):
    old = pd.read_csv(OLD_RESULTS)
    def col(c, default):
        return old[c] if c in old else pd.Series(default, index=old.index)
    m = ((old["ckpt"] == "ema") & (old["epochs"] == EPOCHS) & (col("stem", "stepped") == "stepped")
         & (col("block_norm", "bn") == "bn") & (col("ese", True).astype(str) == "True")
         & (pd.to_numeric(col("warmup", 20)) == 20) & (col("aug_mix", False).astype(str) == "False")
         & (pd.to_numeric(col("seed", 42)) == 42))
    if REFERENCE == "r0.16_full":
        m &= (old["model"] == "cspconvnext") & ((pd.to_numeric(old["csp_skip_ratio"], errors="coerce") - 0.16).abs() < 1e-9)
    else:
        m &= (old["model"] == "convnext_ours")
    ref = old[m].drop_duplicates("fold", keep="last").assign(variant=REFERENCE)
    print(f"Pembanding {REFERENCE} dari CSV lama: {len(ref)} fold, F1 {ref['test_f1_macro'].mean():.4f} "
          f"(naskah: r0.16 = 0.645, kontrol = 0.650)")
    frames.append(ref[["variant", "fold", "test_accuracy", "test_f1_macro", "params_M", "gflops"]])
all_ = pd.concat(frames, ignore_index=True)

ring = all_.groupby("variant").agg(n_fold=("fold", "nunique"), acc=("test_accuracy", "mean"),
    f1_mean=("test_f1_macro", "mean"), f1_std=("test_f1_macro", "std"),
    params_M=("params_M", "first"), gflops=("gflops", "first")).round(4)
print(f"\n=== {EXPERIMENT} ===")
print(ring.to_string())
ring.to_csv(f"/kaggle/working/ringkasan_{EXPERIMENT}.csv")

piv = all_.pivot_table(index="fold", columns="variant", values="test_f1_macro")
if REFERENCE in piv:
    print(f"\n=== Uji berpasangan vs {REFERENCE} (macro F1 per fold) ===")
    for v in [c for c in piv.columns if c != REFERENCE]:
        d = (piv[v] - piv[REFERENCE]).dropna()
        if len(d) >= 2:
            t, p = stats.ttest_rel(piv.loc[d.index, v], piv.loc[d.index, REFERENCE])
            se = d.std(ddof=1) / np.sqrt(len(d)); tc = stats.t.ppf(0.975, len(d) - 1)
            print(f"{v:18s} delta={d.mean():+.4f}  95% CI [{d.mean()-tc*se:+.4f}, {d.mean()+tc*se:+.4f}]  "
                  f"p={p:.4f}  dz={d.mean()/d.std(ddof=1):+.2f}  n={len(d)}")
print("\nUnduh dari tab Output: results_revisi_cv.csv, ringkasan_*.csv, folder preds/")